# Cleanup, variable selection, and QC

In this notebook we will go through the QC procedure to select, and clean the NHANES data

## Initial set up

- Import libraries
- Read data

In [1]:
#### IMPORT MODULES
import modules as md
import clarite
import os
import numpy as np
import pandas as pd

#### SET PATHS
paths  = md.set_project_paths()

#### READ DATA
nhanes = md.load_raw_nhanes(paths[1])
nhanes = md.NhanesData(nhanes)

var_description     = md.load_var_information(paths[1])
var_category        = md.load_var_information(paths[1], description=False)
weights_discovery   = md.load_weights(paths[1])
weights_replication = md.load_weights(paths[1], False)

clarite.describe.summarize(nhanes.data)

rpy2 ModuleSpec(name='rpy2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f5793c8b2d0>, origin='/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2/__init__.py', submodule_search_locations=['/home/tomas/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2'])
41,474 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



## Define phenotypes and covariates


In [2]:
phenotypes = md.get_phenotypes(True, var_description)        
covariates = md.get_covariates()
print(covariates)

	LBXSCRINV = 1/Creatinine (mg/dL)
	URXUCR = Creatinine, urine (mg/dL)
	LBXSCR = Creatinine (mg/dL)
	LBXSATSI = ALT (U/L)
	LBXSAL = Albumin (g/dL)
	URXUMASI = Albumin, urine (mg/L) SI
	URXUMA = Albumin, urine (ug/mL)
	LBXSAPSI = Alkaline phosphotase (U/L)
	LBXSASSI = AST (U/L)
	LBXSC3SI = Bicarbonate (mmol/L)
	LBXSBU = Blood urea nitrogen (mg/dL)
	LBXBAP = Bone alkaline phosphotase (ug/L)
	LBXCPSI = C-peptide: SI(nmol/L)
	LBXCRP = C-reactive protein(mg/dL)
	LBXSCLSI = Chloride (mmol/L)
	LBXSCH = Cholesterol, total (mg/dL)
	LBDHDL = Direct HDL-Cholesterol (mg/dL)
	LBDLDL = LDL-cholesterol (mg/dL)
	LBXFER = Ferritin (ng/mL)
	LBXSGTSI = GGT (U/L)
	LBXSGB = Globulin (g/dL)
	LBXGLU = Glucose, plasma (mg/dL)
	LBXGH = Glycohemoglobin (%)
	LBXHCY = Homocysteine (umol/L)
	LBXSIR = Iron (ug/dL)
	LBXSLDSI = LDH (U/L)
	LBXMMA = Methylmalonic acid (umol/L)
	LBXSOSSI = Osmolality (mOsml/L)
	LBXSPH = Phosphorus (mg/dL)
	LBXSKSI = Potassium (mmol/L)
	LBXEPP = Protoporphyrin (ug/dL RBC)
	LBXSNASI = Sodi

## Keep only adults

In [3]:
nhanes.keep_adults()
clarite.describe.summarize(nhanes.data)

Keeping only adults in the dataset:
-----------------------------------
22,624 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



## Remove participants with missing values in covariates

In [4]:
nhanes.data = clarite.modify.rowfilter_incomplete_obs(nhanes.data, only=covariates)
clarite.describe.summarize(nhanes.data)

Running rowfilter_incomplete_obs
--------------------------------------------------------------------------------
Removed 3,687 of 22,624 observations (16.30%) due to NA values in any of 8 variables
18,937 observations of 1,190 variables
	0 Binary Variables
	0 Categorical Variables
	1,189 Continuous Variables
	1 Unknown-Type Variables



## Keep only those variables that have weights

Also, some weight variables from the replication are not present, remove them as well

In [5]:
remove_vars = md.check_weights(weights_replication)
keep_vars = list(set(nhanes.data.columns))

for i in remove_vars:
    if i in keep_vars:
        print(i + ' is in keep vars')
        keep_vars.remove(i)
        
# Keep only those vars in keep_vars
nhanes.data = clarite.modify.colfilter(nhanes.data, only=keep_vars)

WTSPP2YR is not in weight dataset
WTSPH2YR is not in weight dataset
WTSTH2YR is not in weight dataset
LBXTSH is in keep vars
URX24D is in keep vars
URX25T is in keep vars
URX4FP is in keep vars
URXACE is in keep vars
URXATZ is in keep vars
URXCB3 is in keep vars
URXCCC is in keep vars
URXCMH is in keep vars
URXCPM is in keep vars
URXDEE is in keep vars
URXDIZ is in keep vars
URXDPY is in keep vars
URXMET is in keep vars
URXOPM is in keep vars
URXP08 is in keep vars
URXPAR is in keep vars
URXTCC is in keep vars
Running colfilter
--------------------------------------------------------------------------------
Keeping 1,172 of 1,190 variables:
	0 of 0 binary variables
	0 of 0 categorical variables
	1,171 of 1,189 continuous variables
	1 of 1 unknown variables


## Variable selection and clenup

Here we will:
- Drop any variables that are indeterminant according to the NHANES data dictionary
- Drop variables that don't have 4 year weights
- Drop any non-environmental exposures (examples physical fitness)
- Determine covariates and phenotypes
- From the remainder of the variables split them based on their type
- Manually inspect the definition of ambiguous variables and determine their type
- Merge binary into categorical variables


In [6]:
# SMQ077 and DDB100 have Refused/Don't Know for "7" and "9" values. We will recode them as nan
nhanes.data = clarite.modify.recode_values(nhanes.data, {7: np.nan, 9: np.nan}, only=['SMQ077', 'DBD100'])

# Droping physical fitness measurements, indeterminate variables and age groups
nhanes.drop_unnecesary_vars(var_category)

nhanes.drop_body_measures(var_category, phenotypes+covariates)
#nhanes.drop_disease_vars(var_category)

clarite.describe.summarize(nhanes.data)

Running recode_values
--------------------------------------------------------------------------------
Replaced 11 values from 18,937 observations in 2 variables
Removing physical fitness variables:
-----------------------------------
Removing variables with indeterminate meanings:
-----------------------------------
Removing age categories:
-----------------------------------
Removing body measures variables:
-----------------------------------
18,937 observations of 1,127 variables
	0 Binary Variables
	0 Categorical Variables
	1,126 Continuous Variables
	1 Unknown-Type Variables



## Adjust lipids if on statins

In [7]:
#Adjust for statins (LDL=LDL/0.7, TC=TC/0.8) (2)
nhanes.adjust_lipids(var_description)

Adjusting lipid variables:
-----------------------------------
	ATORVASTATIN_CALCIUM = ATORVASTATIN_CALCIUM
	SIMVASTATIN = SIMVASTATIN
	PRAVASTATIN_SODIUM = PRAVASTATIN_SODIUM
	FLUVASTATIN_SODIUM = FLUVASTATIN_SODIUM


## Separate discovery and replication, and males and females

In [8]:
split_datasets = nhanes.divide_by_sex()

## Remove variables without weights

In [9]:
for i in range(2):
    split_datasets[i].remove_var_without_weights(weights_discovery, extra_vars=phenotypes+covariates)
    clarite.describe.summarize(split_datasets[i].data)
    
for i in range(2,4):
    split_datasets[i].remove_var_without_weights(weights_replication, extra_vars=phenotypes+covariates)
    clarite.describe.summarize(split_datasets[i].data)
    

4,724 observations of 1,022 variables
	0 Binary Variables
	0 Categorical Variables
	1,021 Continuous Variables
	1 Unknown-Type Variables

4,339 observations of 1,022 variables
	0 Binary Variables
	0 Categorical Variables
	1,021 Continuous Variables
	1 Unknown-Type Variables

5,123 observations of 1,022 variables
	0 Binary Variables
	0 Categorical Variables
	1,021 Continuous Variables
	1 Unknown-Type Variables

4,751 observations of 1,022 variables
	0 Binary Variables
	0 Categorical Variables
	1,021 Continuous Variables
	1 Unknown-Type Variables



# Quality control

## Categorize variables

It will also remove all NA variables

In [10]:
for i in range(4):
    split_datasets[i].data = clarite.modify.categorize(split_datasets[i].data)

Running categorize
--------------------------------------------------------------------------------
54 of 1,022 variables (5.28%) are classified as constant (1 unique value).
301 of 1,022 variables (29.45%) are classified as binary (2 unique values).
37 of 1,022 variables (3.62%) are classified as categorical (3 to 6 unique values).
376 of 1,022 variables (36.79%) are classified as continuous (>= 15 unique values).
206 of 1,022 variables (20.16%) were dropped.
	206 variables had zero unique values (all NA).
48 of 1,022 variables (4.70%) were not categorized and need to be set manually.
	47 variables had between 6 and 15 unique values
	1 variables had >= 15 values but couldn't be converted to continuous (numeric) values
Running categorize
--------------------------------------------------------------------------------
63 of 1,022 variables (6.16%) are classified as constant (1 unique value).
274 of 1,022 variables (26.81%) are classified as binary (2 unique values).
40 of 1,022 variable

## Remove constant variables

In [11]:
for l in range(4):
    split_datasets[l].remove_constant_vars(extra_vars=['female', 'male'])

In [12]:
for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 764 variables
	301 Binary Variables
	37 Categorical Variables
	376 Continuous Variables
	48 Unknown-Type Variables

4,339 observations of 724 variables
	274 Binary Variables
	40 Categorical Variables
	370 Continuous Variables
	38 Unknown-Type Variables

5,123 observations of 790 variables
	290 Binary Variables
	44 Categorical Variables
	410 Continuous Variables
	44 Unknown-Type Variables

4,751 observations of 766 variables
	272 Binary Variables
	42 Categorical Variables
	405 Continuous Variables
	45 Unknown-Type Variables



## Examine unknown variables

All unknown variables are continuous, except area which is categorical, and it's at the end of the lists

In [13]:
for i in range(4):
    var_unknown = split_datasets[i].print_unknown_vars(var_description)
    split_datasets[i].data = clarite.modify.make_continuous(split_datasets[i].data, only=var_unknown[:-1])
    split_datasets[i].data = clarite.modify.make_categorical(split_datasets[i].data, only=var_unknown[-1])

	URXUBE = Beryllium, urine (ng/mL)
	URXUPT = Platinum, urine (ng/mL)
	LBDBANO = Basophils number
	URXPPX = 2-isopropoxyphenol (ug/L)
	quantity_drink_per_day = drink per day
	SMD220 = Age started chewing tobacco regularly
	SMD190 = Age started using snuff regularly
	SMD160 = Age started cigar smoking regularly
	SMD130 = Age started pipe smoking regularly
	DRD350BQ = # of times crabs eaten in past 30 days
	DRD350FQ = # of times oysters eaten in past 30 days
	DRD350HQ = # of times shrimp eaten in past 30 days
	DRD370AQ = # of times breaded fish products eaten
	DRD370DQ = # of times catfish eaten in past 30 days
	DRD370EQ = # of times cod eaten in past 30 days
	DRD370FQ = # of times flatfish eaten past 30 days
	DRD370IQ = # of times perch eaten in past 30 days
	DRD370KQ = # of times pollock eaten in past 30 days
	DRD370MQ = # of times salmon eaten in past 30 days
	DRD370NQ = # of times sardines eaten past 30 days
	DRD370TQ = # of times other fish eaten past 30 days
	DRD370UQ = # of times o

## Remove categorical variables

Categorical will not be used, because no beta coefficient is estimated

In [14]:
for l in range(4):
    split_datasets[l].remove_categorical_vars(extra_vars=phenotypes+covariates+['female', 'male'])

In all datasets all unknown variables are continuous, except area which is categorical and it's at the end of the list

In [15]:
for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 727 variables
	301 Binary Variables
	1 Categorical Variables
	423 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 684 variables
	274 Binary Variables
	1 Categorical Variables
	407 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 746 variables
	290 Binary Variables
	1 Categorical Variables
	453 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 724 variables
	272 Binary Variables
	1 Categorical Variables
	449 Continuous Variables
	0 Unknown-Type Variables



## Remove phenotypes that were dropped

Because of the QC, some phenotypes might have been deleted

In [16]:
remove_phenos = []
for i in phenotypes:
    for l in range(4):
        cols = split_datasets[l].data.columns
        if i not in cols:
            print(i + ' was deleted in at least one dataset')
            remove_phenos.append(i)

# Remove the deleted phenos from all datasets
for i in remove_phenos:
    phenotypes.remove(i)
    for l in range(4):
        if i in split_datasets[l].data.columns:
            split_datasets[l].data = split_datasets[l].data.drop(columns=i)

LBXFER was deleted in at least one dataset
LBXEPP was deleted in at least one dataset
LBXTIB was deleted in at least one dataset
LBDPCT was deleted in at least one dataset
LBXIRN was deleted in at least one dataset


## QC

We will remove variables that have less than 200 non-nan values

In [17]:
for i in range(4):
    split_datasets[i].data = clarite.modify.colfilter_min_n(split_datasets[i].data,
                                                            skip=phenotypes + covariates + ['male', 'female'])

Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 296 of 301 binary variables
	Removed 33 (11.15%) tested binary variables which had less than 200 non-null values.
Testing 0 of 1 categorical variables
Testing 363 of 418 continuous variables
	Removed 17 (4.68%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 269 of 274 binary variables
	Removed 28 (10.41%) tested binary variables which had less than 200 non-null values.
Testing 0 of 1 categorical variables
Testing 347 of 402 continuous variables
	Removed 12 (3.46%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 285 of 290 binary variables
	Removed 28 (9.82%) tested binary variables which had less than 200 non-n

In [18]:
for i in range(4):
    split_datasets[i].data = clarite.modify.colfilter_min_cat_n(split_datasets[i].data, 
                                                                skip=[c for c in covariates + phenotypes + ['male', 'female'] if c in split_datasets[i].data.columns] )

Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 263 of 268 binary variables
	Removed 208 (79.09%) tested binary variables which had a category with less than 200 values.
Testing 0 of 1 categorical variables
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 241 of 246 binary variables
	Removed 193 (80.08%) tested binary variables which had a category with less than 200 values.
Testing 0 of 1 categorical variables
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 257 of 262 binary variables
	Removed 200 (77.82%) tested binary variables which had a category with less than 200 values.
Testing 0 of 1 categorical variables
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 247 of 252 binary variables
	Removed 194 (78.54%) teste

In [19]:
# 90percent zero filter
for i in range(4):
    split_datasets[i].data = clarite.modify.colfilter_percent_zero(split_datasets[i].data)

Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 401 of 401 continuous variables
	Removed 23 (5.74%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 390 of 390 continuous variables
	Removed 20 (5.13%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 416 of 416 continuous variables
	Removed 12 (2.88%) tested continuous variables which were equal to zero in at least 90.00% of non-NA observations.
Running colfilter_percent_zero
--------------------------------------------------------------------------------
Testing 426 of 426 continuous variables
	Removed 26 (6.10%) tested continuous variab

In [20]:
for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 441 variables
	60 Binary Variables
	1 Categorical Variables
	378 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 426 variables
	53 Binary Variables
	1 Categorical Variables
	370 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 469 variables
	62 Binary Variables
	1 Categorical Variables
	404 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 461 variables
	58 Binary Variables
	1 Categorical Variables
	400 Continuous Variables
	0 Unknown-Type Variables



In [21]:
# Take note of which variables were differently typed in each dataset
print("Correcting differences in variable types between discovery and replication")
# Merge current type series
dtypes = pd.DataFrame({'discovery_females':clarite.describe.get_types(split_datasets[0].data),
                       'discovery_males':clarite.describe.get_types(split_datasets[1].data),
                       'replication_females':clarite.describe.get_types(split_datasets[2].data),
                       'replication_males':clarite.describe.get_types(split_datasets[3].data),
                       })

for i,l in dtypes.iterrows():
    m = set(l)
    m = {x for x in m if x==x}
    if len(m) > 1:
        print('There is no agreement in here ... ')
        print(i)
        print(l)

# There are no differences

Correcting differences in variable types between discovery and replication


## Keeping variables that are in all datasets

In [22]:
all_vars = set(list(split_datasets[0].data)) & \
           set(list(split_datasets[1].data)) & \
           set(list(split_datasets[2].data)) & \
           set(list(split_datasets[3].data))

for i in range(4):
    split_datasets[i].data = split_datasets[i].data[all_vars]
    
print(f"{len(all_vars)} variables in common")

369 variables in common


## Inspect variables

Inpecting plots before transformations

In [23]:
text = ['_d_f_raw', '_d_m_raw', '_r_f_raw', '_r_m_raw']
plot_paths = os.path.join(paths[2], 'Plots', 'Inspect')
for i in range(4):
    split_datasets[i].plot_variables(plot_paths, phenotypes, covariates, text[i])

Generating a 4 page PDF for 41 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 41 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 41 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 41 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 325 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables


## Transform continuous variables

All continuous variables that show greater skew than -0.5 to 0.5. First we will need to merge back the datasets

In [23]:
nhanes = pd.concat([split_datasets[0].data, 
                    split_datasets[1].data, 
                    split_datasets[2].data, 
                    split_datasets[3].data])

nhanes = md.NhanesData(nhanes)
clarite.describe.summarize(nhanes.data)

18,937 observations of 369 variables
	40 Binary Variables
	1 Categorical Variables
	328 Continuous Variables
	0 Unknown-Type Variables



In [25]:
nhanes.log_transform_skewed()

Tranforming 294 positevely skewed, and 6 negatevely skewed variables
for DR1TCOPP c is 0.0002
for DR1TPFAT c is 0.0001
for SMD057 c is 0.01
for COPPER_mg c is 5.66666661e-06
for supplement_count c is 0.01
for SMQ050 c is 0.01
for DR1TS180 c is 7.5e-05
for SMD075 c is 0.01
for LBXME c is 0.0001
for VITAMIN_B_12_mcg c is 0.00066666666
for DR1TP184 c is 5e-06
for THIAMIN_mg c is 8e-05
for DR1TP183 c is 2e-05
for DR1TNIAC c is 0.0008
for DR1TP204 c is 5e-06
for DR1TPHOS c is 0.0002
for DR1TLYCO c is 0.005
for VITAMIN_A_IU c is 0.0833333333333333
for SELENIUM_mcg c is 0.00233333333
for CALCIUM_mg c is 0.005
for DR1TSELE c is 0.0005
for DR1TVARA c is 0.01
for DR1TTFAT c is 0.0001
for DR1TS080 c is 5e-06
for DR1TZINC c is 0.0004
for DR1TP226 c is 5e-06
for DR1TCARB c is 0.0118
for DR1TS040 c is 5e-06
for DR1TALCO c is 0.0001
for DR1TVC c is 0.0001
for DR1TP225 c is 5e-06
for DR1TCALC c is 0.0592
for LBXEOPCT c is 0.001
for VITAMIN_B_6_mg c is 0.000166
for DR1TM221 c is 5e-06
for IRON_mg c is 

## Normalize variables

In [27]:
# Merge them to normalize across sexes while keeping the previous variable categories
var_types = clarite.describe.get_types(nhanes.data)
var_continuous  = list(var_types[var_types == 'continuous'].index)
var_binary      = var_types[var_types == 'binary'].index
var_categorical = var_types[var_types == 'categorical'].index
var_constant    = var_types[var_types == 'constant'].index

In [28]:
# Normalize only continuous that are not covariates
for i in covariates:
    if i in var_continuous:
        var_continuous.remove(i)

# Get the columns that are not in the var_continuous list
cols = list(nhanes.data.columns)
for i in var_continuous:
    cols.remove(i)

In [29]:
from sklearn.preprocessing import normalize

nhanes_c = nhanes.data[var_continuous] # nhanes continuous without covariates
nhanes_e = nhanes.data[cols] # nhanes everything else

nhanes_c = (nhanes_c - nhanes_c.min()) / (nhanes_c.max() - nhanes_c.min()) # Normalizing with pandas ignoring NaN

In [30]:
# Merge back
nhanes = clarite.modify.merge_variables(nhanes_e, nhanes_c, how='inner')
nhanes = md.NhanesData(nhanes)
clarite.describe.summarize(nhanes.data)

Running merge_variables
--------------------------------------------------------------------------------
inner Merge:
	left = 18,937 observations of 44 variables
	right = 18,937 observations of 325 variables
Kept 18,937 observations of 369 variables.
18,937 observations of 369 variables
	40 Binary Variables
	1 Categorical Variables
	328 Continuous Variables
	0 Unknown-Type Variables



## Inspect transformed data before spliting

In [31]:
plot_paths = os.path.join(paths[2], 'Plots', 'Inspect')
nhanes.plot_variables(plot_paths, phenotypes, covariates, '_total_transformed')

Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 328 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables


## Spliting dataset back

In [32]:
split_datasets = nhanes.divide_by_sex()

for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 369 variables
	40 Binary Variables
	1 Categorical Variables
	328 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 369 variables
	40 Binary Variables
	1 Categorical Variables
	328 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 369 variables
	40 Binary Variables
	1 Categorical Variables
	328 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 369 variables
	40 Binary Variables
	1 Categorical Variables
	328 Continuous Variables
	0 Unknown-Type Variables



## Inspect tranformed variables split

In [33]:
text = ['_d_f_transformed', '_d_m_transformed', '_r_f_transformed', '_r_m_transformed']
plot_paths = os.path.join(paths[2], 'Plots', 'Inspect')
for i in range(4):
    split_datasets[i].plot_variables(plot_paths, phenotypes, covariates, text[i])

Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 328 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 328 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 328 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables
Generating a 4 page PDF for 40 variables
Generating a 1 page PDF for 1 variables
Generating a 28 page PDF for 328 variables
Generating a 5 page PDF for 53 variables
Generating a 1 page PDF for 8 variables


## Remove sex and unnecesary variables

In [34]:
remove_vars = ['male', 'female', 'white'] # The last one is not in the replication weight
for i in range(4):
    split_datasets[i].data = split_datasets[i].data.drop(columns=remove_vars)

In [35]:
for i in range(4):
    clarite.describe.summarize(split_datasets[i].data)

4,724 observations of 366 variables
	39 Binary Variables
	1 Categorical Variables
	326 Continuous Variables
	0 Unknown-Type Variables

4,339 observations of 366 variables
	39 Binary Variables
	1 Categorical Variables
	326 Continuous Variables
	0 Unknown-Type Variables

5,123 observations of 366 variables
	39 Binary Variables
	1 Categorical Variables
	326 Continuous Variables
	0 Unknown-Type Variables

4,751 observations of 366 variables
	39 Binary Variables
	1 Categorical Variables
	326 Continuous Variables
	0 Unknown-Type Variables



## Save cleaned data

In [36]:
os.chdir(paths[2])
savef = ['Discovery_Females', 'Discovery_Males', 'Replication_Females', 'Replication_Males']
for i in range(4):
    split_datasets[i].data.to_csv('CleanData_' + savef[i] + '.csv' )